In [ ]:
# import pandas
import pandas as pd

# load the raw csv file
df = pd.read_csv("surat_house_price.csv")

# display the raw data
print("Shape:", df.shape)
df.head()

In [ ]:
# check column names and data types
df.info()

In [ ]:
# check missing values per column
df.isnull().sum()

In [ ]:
# check number of duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# drop duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# display data after removing duplicates
print("Shape after dropping duplicates:", df.shape)
df.head()

In [ ]:
# strip extra spaces from column names
df.columns = df.columns.str.strip()

# display updated column names
df.columns

In [ ]:
# strip leading/trailing whitespace from all text columns
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].str.strip()

# display data after trimming whitespace
df.head()

In [ ]:
# clean price_per_sqft column: remove currency symbol, commas and text, keep number only
df["price_per_sqft"] = (
    df["price_per_sqft"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("per sqft", "", regex=False)
    .str.strip()
)
df["price_per_sqft"] = pd.to_numeric(df["price_per_sqft"], errors="coerce")

# display data after cleaning price_per_sqft
df[["price_per_sqft"]].head(10)

In [ ]:
# clean price column: remove currency symbol and convert Lac/Cr to a single unit (Lac)
df["price"] = df["price"].astype(str).str.replace("₹", "", regex=False).str.strip()

# extract the numeric value and the unit (Lac or Cr)
df["price_value"] = df["price"].str.extract(r"([\d.]+)").astype(float)
df["price_unit"] = df["price"].str.extract(r"(Lac|Cr)")

# convert everything to Lac (1 Cr = 100 Lac)
df["price_in_lac"] = df.apply(
    lambda row: row["price_value"] * 100 if row["price_unit"] == "Cr" else row["price_value"],
    axis=1,
)

# drop the helper columns now that price_in_lac is ready
df = df.drop(columns=["price", "price_value", "price_unit"])

# display data after cleaning price
df[["price_in_lac"]].head(10)

In [ ]:
# clean square_feet column: extract numeric value and unit (sqft or sqm)
df["area_value"] = df["square_feet"].str.extract(r"([\d.]+)").astype(float)
df["area_unit"] = df["square_feet"].str.extract(r"(sqft|sqm)")

# convert sqm to sqft so every row uses the same unit (1 sqm = 10.7639 sqft)
df["area_sqft"] = df.apply(
    lambda row: row["area_value"] * 10.7639 if row["area_unit"] == "sqm" else row["area_value"],
    axis=1,
)

# drop the helper columns now that area_sqft is ready
df = df.drop(columns=["square_feet", "area_value", "area_unit"])

# display data after cleaning square_feet
df[["area_sqft"]].head(10)

In [ ]:
# clean description column: remove the trailing 'Read more' text
df["description"] = df["description"].astype(str).str.replace("Read more", "", regex=False).str.strip()

# display data after cleaning description
df[["description"]].head()

In [ ]:
# extract number of BHK from property_name into a new column
df["bhk"] = df["property_name"].str.extract(r"(\d+)\s*BHK").astype(float)

# display data after extracting bhk
df[["property_name", "bhk"]].head(10)

In [ ]:
# fill missing values in remaining text columns with 'Not Specified'
remaining_text_cols = ["areaWithType", "transaction", "status", "floor", "furnishing", "facing"]
for col in remaining_text_cols:
    df[col] = df[col].fillna("Not Specified")

# display missing values after filling
df.isnull().sum()

In [ ]:
# drop rows where price or area could not be determined
df = df.dropna(subset=["price_in_lac", "area_sqft"]).reset_index(drop=True)

# display data after dropping rows with missing key values
print("Shape after dropping rows with missing price/area:", df.shape)
df.head()

In [ ]:
# final check of cleaned dataset
df.info()
df.describe()

In [ ]:
# save the cleaned dataset to a new csv file
df.to_csv("surat_house_price_cleaned.csv", index=False)

# display the final cleaned data
df.head(10)